In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import save_image
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm

# =========================
# CONFIG
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

Z_DIM = 256
IN_CHANNELS = 256
CHANNELS_IMG = 3

LEARNING_RATE = 1e-3
LAMBDA_GP = 10

PROGRESSIVE_EPOCHS = [5, 5, 8, 8, 10]  # 4→64
BATCH_SIZES = [128, 128, 64, 32, 16]

FACTORS = [1, 1, 1/2, 1/4, 1/8]

# =========================
# UTILS
# =========================
class PixelNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.eps = 1e-8

    def forward(self, x):
        return x / torch.sqrt(torch.mean(x**2, dim=1, keepdim=True) + self.eps)


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, 1, 1),
            nn.LeakyReLU(0.2),
            PixelNorm(),
            nn.Conv2d(out_c, out_c, 3, 1, 1),
            nn.LeakyReLU(0.2),
            PixelNorm()
        )

    def forward(self, x):
        return self.block(x)

# =========================
# GENERATOR
# =========================
class Generator(nn.Module):
    def __init__(self, z_dim, in_channels, img_channels):
        super().__init__()

        self.initial = nn.Sequential(
            nn.ConvTranspose2d(z_dim, in_channels, 4, 1, 0),
            nn.LeakyReLU(0.2),
            PixelNorm()
        )

        self.prog_blocks = nn.ModuleList([])
        self.rgb_layers = nn.ModuleList([nn.Conv2d(in_channels, img_channels, 1)])

        for i in range(len(FACTORS) - 1):
            in_c = int(in_channels * FACTORS[i])
            out_c = int(in_channels * FACTORS[i + 1])
            self.prog_blocks.append(ConvBlock(in_c, out_c))
            self.rgb_layers.append(nn.Conv2d(out_c, img_channels, 1))

    def fade_in(self, alpha, upscaled, generated):
        return torch.tanh(alpha * generated + (1 - alpha) * upscaled)

    def forward(self, x, alpha, steps):
        out = self.initial(x)

        if steps == 0:
            return torch.tanh(self.rgb_layers[0](out))

        for step in range(steps):
            upscaled = nn.functional.interpolate(out, scale_factor=2, mode="nearest")
            out = self.prog_blocks[step](upscaled)

        final_upscaled = self.rgb_layers[steps - 1](upscaled)
        final_out = self.rgb_layers[steps](out)

        return self.fade_in(alpha, final_upscaled, final_out)

# =========================
# DISCRIMINATOR
# =========================
class Discriminator(nn.Module):
    def __init__(self, in_channels, img_channels):
        super().__init__()

        self.prog_blocks = nn.ModuleList([])
        self.rgb_layers = nn.ModuleList([])

        for i in range(len(FACTORS)):
            in_c = int(in_channels * FACTORS[i])
            self.rgb_layers.append(nn.Conv2d(img_channels, in_c, 1))

        for i in range(len(FACTORS) - 1, 0, -1):
            in_c = int(in_channels * FACTORS[i])
            out_c = int(in_channels * FACTORS[i - 1])
            self.prog_blocks.append(ConvBlock(in_c, out_c))

        self.final = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, 1, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(in_channels, 1, 4, 1, 0)
        )

    def fade_in(self, alpha, downscaled, out):
        return alpha * out + (1 - alpha) * downscaled

    def forward(self, x, alpha, steps):
        cur_step = len(self.prog_blocks) - steps

        out = self.rgb_layers[steps](x)

        if steps == 0:
            return self.final(out).view(out.shape[0], -1)

        downscaled = nn.functional.avg_pool2d(x, 2)
        downscaled = self.rgb_layers[steps - 1](downscaled)

        out = self.prog_blocks[cur_step](out)
        out = nn.functional.avg_pool2d(out, 2)   # 🔥 IMPORTANT FIX
        out = self.fade_in(alpha, downscaled, out)

        for step in range(cur_step + 1, len(self.prog_blocks)):
            out = self.prog_blocks[step](out)
            out = nn.functional.avg_pool2d(out, 2)

        return self.final(out).view(out.shape[0], -1)

# =========================
# DATA LOADER
# =========================
def get_loader(image_size):
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    dataset = torchvision.datasets.CelebA(
        root="data",
        split="train",
        transform=transform,
        download=True
    )

    batch_size = BATCH_SIZES[int(torch.log2(torch.tensor(image_size))) - 2]

    return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# =========================
# GRADIENT PENALTY
# =========================
def gradient_penalty(critic, real, fake, alpha, step):
    B, C, H, W = real.shape
    epsilon = torch.rand((B, 1, 1, 1), device=DEVICE).repeat(1, C, H, W)

    interpolated = real * epsilon + fake * (1 - epsilon)
    interpolated.requires_grad_(True)

    mixed_scores = critic(interpolated, alpha, step)

    gradient = torch.autograd.grad(
        inputs=interpolated,
        outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True
    )[0]

    gradient = gradient.view(gradient.shape[0], -1)
    gp = torch.mean((gradient.norm(2, dim=1) - 1) ** 2)
    return gp

# =========================
# TRAIN
# =========================
def show_images(tensor, title=""):
    img = tensor[:16] * 0.5 + 0.5
    grid = torchvision.utils.make_grid(img, nrow=4)
    npimg = grid.cpu().numpy()
    
    plt.figure(figsize=(6,6))
    plt.title(title)
    plt.imshow(np.transpose(npimg, (1,2,0)))
    plt.axis("off")
    plt.show()
def train():
    gen = Generator(Z_DIM, IN_CHANNELS, CHANNELS_IMG).to(DEVICE)
    disc = Discriminator(IN_CHANNELS, CHANNELS_IMG).to(DEVICE)

    opt_gen = optim.Adam(gen.parameters(), lr=LEARNING_RATE, betas=(0.0, 0.99))
    opt_disc = optim.Adam(disc.parameters(), lr=LEARNING_RATE, betas=(0.0, 0.99))

    fixed_noise = torch.randn(16, Z_DIM, 1, 1).to(DEVICE)

    step = 0

    for num_epochs in PROGRESSIVE_EPOCHS:
        image_size = 4 * 2 ** step
        loader = get_loader(image_size)

        print(f"\n🔥 Training {image_size}x{image_size}")

        alpha = 0

        for epoch in range(num_epochs):
            loop = tqdm(loader, leave=True)

            for batch_idx, (real, _) in enumerate(loop):
                real = real.to(DEVICE)
                cur_batch_size = real.shape[0]

                noise = torch.randn(cur_batch_size, Z_DIM, 1, 1).to(DEVICE)
                fake = gen(noise, alpha, step)

                # ---- Train Discriminator ----
                disc_real = disc(real, alpha, step)
                disc_fake = disc(fake.detach(), alpha, step)

                gp = gradient_penalty(disc, real, fake, alpha, step)

                loss_disc = -(torch.mean(disc_real) - torch.mean(disc_fake)) + LAMBDA_GP * gp

                opt_disc.zero_grad()
                loss_disc.backward()
                opt_disc.step()

                # ---- Train Generator ----
                noise2 = torch.randn(cur_batch_size, Z_DIM, 1, 1).to(DEVICE)
                fake2 = gen(noise2, alpha, step)
                
                gen_fake = disc(fake2, alpha, step)
                loss_gen = -torch.mean(gen_fake)
                opt_gen.zero_grad()
                loss_gen.backward()
                opt_gen.step()

                alpha += cur_batch_size / ((num_epochs * len(loader)))
                alpha = min(alpha, 1)

                loop.set_postfix(loss_D=loss_disc.item(), loss_G=loss_gen.item())
            # ---- SAVE IMAGES ----
            with torch.no_grad():
                fake = gen(fixed_noise, alpha, step)
                os.makedirs("outputs", exist_ok=True)
            
                save_image(fake * 0.5 + 0.5,
                           f"outputs/res_{image_size}_epoch_{epoch}.png")
            
                if epoch % 2 == 0:  # show every epoch
                    show_images(fake, title=f"{image_size}x{image_size} | epoch {epoch} | alpha {alpha:.2f}")

        step += 1


if __name__ == "__main__":
    train()

FileURLRetrievalError: Failed to retrieve file url:

	Too many users have viewed or downloaded this file recently. Please
	try accessing the file again later. If the file you are trying to
	access is particularly large or is shared with many people, it may
	take up to 24 hours to be able to view or download the file. If you
	still can't access a file after 24 hours, contact your domain
	administrator.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=0B7EVK8r0v71pZjFTYXZWM3FlRnM

but Gdown can't. Please check connections and permissions.